# Pipeline 04Gb: global JSON → text (G2a, per feature)

For each `(model, feature)`: the LLM receives the feature's global effect **curve as a
JSON payload** and writes a `[EFFECT] / [IMPORTANCE] / [RECOMMENDATION]` description.
Unit = feature (not instance). System prompt = shared core + JSON handover from
`prompts/global_feature.md` ({{MODEL}} filled, cached). Output:
`results/global/json_{model}_{feature}.json`. Resumable (skips existing files).
Compared against the deterministic baseline `04Ga` and the Vision/Tool-Use pipelines.

In [ ]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR, GLOBAL_RESULTS_SUBDIR,
    list_global_features, build_feature_json_payload, assemble_global_system_prompt,
    build_global_record, run_resumable_global_generation,
)
from utils.llm import ask_text, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION
XAI_MODELS = ['xgb', 'ebm']

OUT_DIR = RESULTS_DIR / GLOBAL_RESULTS_SUBDIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)

print(f'LLM model: {MODEL}')
print(f'Features:  {len(FEATURES)} x {len(XAI_MODELS)} models = {len(FEATURES) * len(XAI_MODELS)} calls')
print(f'Output:    {OUT_DIR}')

In [ ]:
# System prompt per model: shared core + JSON handover, {{MODEL}} filled. Identical
# across json/vision/tooluse except the handover block (the modality-comparison
# invariant). Cached via cache_system=True -> one cache entry per model.
SYSTEM = {m: assemble_global_system_prompt('json', m, prompts_dir=PROMPTS_DIR) for m in XAI_MODELS}
print(SYSTEM['ebm'][:500], '\n...')

In [ ]:
# Resume/persistence via utils.run_resumable_global_generation (skip if exists,
# idempotent, lossless). Record schema = utils.build_global_record (scope='global').
# generate returns None on error -> the (model, feature) stays open for the next run.

def generate_json(model_name, feature, gen_idx):
    payload = build_feature_json_payload(model_name, feature, explanations_dir=EXPLANATIONS_DIR)
    user = (
        f'Describe the global effect of the feature "{feature}" on hourly bike demand.\n\n'
        + json.dumps(payload, indent=2)
    )
    t0 = time.time()
    try:
        response = ask_text(user, system=SYSTEM[model_name], model=MODEL,
                            max_tokens=MAX_TOKENS, cache_system=True)
    except Exception as e:
        print(f'  [ERROR] {model_name} {feature}: {type(e).__name__}: {e} -> skip')
        return None
    elapsed = time.time() - t0

    text  = strip_scratchpad(response['content'][0]['text'])
    usage = response.get('usage', {})
    record = build_global_record(
        form='json', model_name=model_name, feature=feature, explanation=text,
        usage=usage, llm_model=MODEL, loss_key=LOSS_KEY, elapsed_s=round(elapsed, 2),
    )
    u = record['usage']
    print(f"  {model_name.upper()} {feature:11} in={u['input_tokens']} out={u['output_tokens']} "
          f"cache={u.get('cache_read_input_tokens', 0)} t={elapsed:.1f}s")
    return record


results = run_resumable_global_generation(
    form='json', model_names=XAI_MODELS, features=FEATURES,
    out_dir=OUT_DIR, generate=generate_json,
)

totals = {k: sum(r['usage'].get(k, 0) for r in results)
          for k in ('input_tokens', 'output_tokens', 'cache_read_input_tokens')}
print(f"\nTotal: {totals}  ({len(results)} descriptions)")